# AION-U1 Pilot-Training (Colab T4)

Zwei Läufe: Training und danach dieselbe Messung, die auch die Lehrer erfasst.
Ergebnis liegt in `reports/eval-pilot.json`; der Adapter in `artifacts/sft/pilot_open/final_model/`.

In [ ]:
# 1) Repo
!git clone -q https://github.com/Reznap87/aion-u1.git
%cd aion-u1
!pip install -q -e '.[train]'
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 2) Config-Verifikation. sft_lora_open_v4.yaml ist bewusst fp32:
#    T4 = sm_75, kein bf16. fp16 ohne Scaler liefert NaN-Gradienten und
#    einen 'completed'-Lauf mit lora_B == 0. Bei 66 Saetzen braucht es
#    ausserdem Akkumulation 4 / 3 Epochen / LR 1e-4 / warmup 0.05 und
#    kein load_best_model_at_end, solange eval_loss nicht belastbar ist.
import pathlib, yaml
cfg = yaml.safe_load(pathlib.Path('training/configs/sft_lora_open_v4.yaml').read_text(encoding='utf-8'))
assert cfg['model']['dtype'] == 'float32', cfg['model']['dtype']
assert cfg['training']['fp16'] is False and cfg['training']['bf16'] is False
assert cfg['training']['gradient_accumulation_steps'] == 4
assert cfg['training']['num_train_epochs'] == 3
assert cfg['training']['load_best_model_at_end'] is False
print('Config ok: fp32, 3 Epochen, Akkumulation 4, LR', cfg['training']['learning_rate'])


In [ ]:
# 3) Training (66 Trainingssaetze, LoRA r=16)
!aion-u1-train-sft --config training/configs/sft_lora_open_v4.yaml
import pathlib
if not pathlib.Path('artifacts/sft/pilot_open/final_model/adapter_model.safetensors').is_file():
    raise SystemExit('Kein Adapter erzeugt — siehe Fehlermeldung von Zelle 3')

# lora_B startet exakt bei null. Bleibt es null, hat der Lauf nichts gelernt
# (typisch: NaN-Gradienten durch fp16-Optimierzustaende).
from safetensors.torch import load_file
staendew = load_file('artifacts/sft/pilot_open/final_model/adapter_model.safetensors')
b_max = max(float(v.abs().max()) for k, v in staendew.items() if k.endswith('lora_B.weight'))
if b_max == 0.0:
    raise SystemExit(f'lora_B ist null (max={b_max}) — nichts gelernt, siehe loss/grad_norm im Manifest')
print('Adapter liegt vor; lora_B max:', b_max)


In [ ]:
# 4) Baseline (gleiche Suite, ohne Adapter) — fuer den Vergleich
!aion-u1-evaluate-model --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --suite evaluations/capability_suite/benchmark_hard_v1.yaml \
  --device cuda --output reports/eval-baseline.json

In [ ]:
# 5) Messung des Adapter-Modells (harte Suite, dieselbe Aggregation)
!aion-u1-evaluate-model --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --adapter artifacts/sft/pilot_open/final_model \
  --suite evaluations/capability_suite/benchmark_hard_v1.yaml \
  --device cuda --output reports/eval-pilot.json

In [ ]:
# 6) Ergebnis lesen
import json
for name in ('eval-baseline', 'eval-pilot'):
    try:
        d = json.load(open(f'reports/{name}.json', encoding='utf-8'))
        s = list(d['scores'].values())[0]
        print(f"{name:14} index={s.get('overall_score')} gemessen={s.get('measured_score')} coverage={s.get('coverage')}")
    except FileNotFoundError:
        print(name, 'fehlt')
print('Zielmarke GOAL.md: 0.847 (beste Lehrer, harte Suite)')

In [ ]:
# 7) Artefakte sichern (Adapter herunterladen, bevor die Session endet)
import pathlib
if not pathlib.Path('artifacts/sft/pilot_open').is_dir():
    raise SystemExit('artifacts/sft/pilot_open fehlt — Training (Zelle 3) pruefen')
!zip -q -r sft_pilot_open.zip artifacts/sft/pilot_open
from google.colab import files
files.download('sft_pilot_open.zip')